# YOLOv8 Baseline - Colab

Train, evaluate, and run inference with YOLOv8n on Google Drive data.
Setup cells mirror `colab_template.ipynb`.

In [ ]:
REPO = "road-damage-detection"

# Clone the repository (skip if already cloned)
!test -d $REPO || git clone https://github.com/orzmik/road-damage-detection.git
%cd $REPO

# Install dependencies needed for Colab runs
!pip -q install ultralytics wandb

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [ ]:
# My Drive:
DRIVE_ROOT = "MyDrive/road-damage-detection"

# Shared drive (teammates):
# DRIVE_ROOT = "Shareddrives/<TeamDrive>/road-damage-detection"

In [ ]:
from src.config.colab.drive import DriveConfig, ensure_drive_paths, mount_drive

mount_drive()

drive_cfg = DriveConfig(drive_root=DRIVE_ROOT)
paths = ensure_drive_paths(drive_cfg)

print(f"Drive root:       {paths.root}")
print(f"Processed data:   {paths.processed_yolo}")
print(f"Models:           {paths.models}")
print(f"Training runs:    {paths.runs}")
print(f"Wroclaw images:   {paths.wroclaw_images}")

## Training

In [ ]:
import os
import wandb
from getpass import getpass

if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass("Enter your W&B API key: ")

wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
from src.config.wandb import WandbConfig
from src.config.yolo import create_yolo_data_yaml, train_yolo

data_yaml = create_yolo_data_yaml(
    paths.data / "road_damage_drive.yaml",
    paths.processed_yolo,
)

wandb_cfg = WandbConfig(
    project="road-damage-classification",
    entity="project-nn",
    name="yolov8n_colab",
    job_type="train",
    config={
        "epochs": 50,
        "imgsz": 640,
        "batch": 16,
    },
)

train_out = train_yolo(
    weights="yolov8n.pt",
    data_yaml=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab",
    wandb_cfg=wandb_cfg,
)

train_out.save_dir

## Evaluation (test set)

In [ ]:
from src.config.yolo import evaluate_yolo

best_weights = train_out.save_dir / "weights" / "best.pt"
metrics = evaluate_yolo(
    weights=best_weights,
    data_yaml=data_yaml,
    split="test",
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab_test",
)
metrics

## Inference (Wroclaw images)

In [ ]:
from src.config.yolo import predict_yolo

predict_results = predict_yolo(
    weights=best_weights,
    source=paths.wroclaw_images,
    device=0,
    project_dir=paths.runs,
    run_name="yolov8n_colab_infer",
    save=True,
)

predict_results[:2]